In [10]:
import numpy as np
from logistic_regression_torch import LogisticRegression2
import torch
import torch.nn.functional as F
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import pandas as pd

device_mps = torch.device("mps" if torch.backends.mps.is_available() else "cpu")



In [32]:
data = np.genfromtxt('data/toydata.txt', delimiter='\t')
x = data[:, :2].astype(np.float32)
y = data[:, 2].astype(np.int64)



In [33]:
np.random.seed(123)
idx = np.arange(y.shape[0])
np.random.shuffle(idx)
X_test, y_test = x[idx[:25]], y[idx[:25]]
X_train, y_train = x[idx[25:]], y[idx[25:]]
mu, std = np.mean(X_train, axis=0), np.std(X_train, axis=0)
X_train, X_test = (X_train - mu) / std, (X_test - mu) / std


In [35]:
scaler =StandardScaler()
x_train_norm =scaler.fit_transform(X_train)
x_test_norm = scaler.transform(X_test)
X_train_tensor = torch.tensor(x_train_norm,dtype=torch.float32,device=device_mps)
y_train_tensor = torch.tensor(y_train,dtype=torch.float32,device=device_mps).view(-1)

In [50]:
model2 = LogisticRegression2(num_features=2).to(device_mps)
optimizer = torch.optim.SGD(model2.parameters(),lr =0.1)

In [51]:
from torch.utils.data import Dataset

class SimpleDataset(Dataset):
    def __init__(self,X,y):
        # features (X) and labels (y)
        self.X = X
        self.y = y

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

dataset = SimpleDataset(X_train_tensor,y_train_tensor)


In [52]:
from torch.utils.data import DataLoader

loader = DataLoader(dataset, batch_size=10, shuffle=True)

In [53]:
def comp_accuracy(label,pred_label):
    pred_label= torch.where(pred_label>0.5,1,0).view(-1)
    acu = torch.sum(pred_label==label.view(-1)).float()/label.shape[0]
    return acu

num_epchos =30
for epoch in range(num_epchos):
    for x_batch,y_batch in loader:
        out = model2.forward(X_train_tensor)
        loss = F.binary_cross_entropy(out,y_train_tensor.view(-1,1),reduction='sum')
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        acc = comp_accuracy(y_train_tensor,out)
        pred_probas = model2(X_train_tensor)
        acc = comp_accuracy(y_train_tensor, pred_probas)
        print('Epoch: %03d' % (epoch + 1), end="")
        print(' | Train ACC: %.3f' % acc, end="")
        print(' | Cost: %.3f' % F.binary_cross_entropy(out,y_train_tensor.view(-1,1)))


Epoch: 001 | Train ACC: 0.973 | Cost: 0.693
Epoch: 001 | Train ACC: 0.973 | Cost: 0.055
Epoch: 001 | Train ACC: 0.973 | Cost: 0.053
Epoch: 001 | Train ACC: 0.973 | Cost: 0.051
Epoch: 001 | Train ACC: 0.973 | Cost: 0.049
Epoch: 001 | Train ACC: 0.973 | Cost: 0.048
Epoch: 001 | Train ACC: 0.973 | Cost: 0.047
Epoch: 001 | Train ACC: 0.973 | Cost: 0.046
Epoch: 002 | Train ACC: 0.973 | Cost: 0.045
Epoch: 002 | Train ACC: 0.987 | Cost: 0.044
Epoch: 002 | Train ACC: 0.987 | Cost: 0.043
Epoch: 002 | Train ACC: 0.987 | Cost: 0.042
Epoch: 002 | Train ACC: 0.987 | Cost: 0.041
Epoch: 002 | Train ACC: 0.987 | Cost: 0.041
Epoch: 002 | Train ACC: 0.987 | Cost: 0.040
Epoch: 002 | Train ACC: 0.987 | Cost: 0.039
Epoch: 003 | Train ACC: 1.000 | Cost: 0.039
Epoch: 003 | Train ACC: 1.000 | Cost: 0.038
Epoch: 003 | Train ACC: 1.000 | Cost: 0.038
Epoch: 003 | Train ACC: 1.000 | Cost: 0.037
Epoch: 003 | Train ACC: 1.000 | Cost: 0.036
Epoch: 003 | Train ACC: 1.000 | Cost: 0.036
Epoch: 003 | Train ACC: 1.000 | 